In [1]:
# dry run: run the full pipeline and build the backend records (nothing is sent yet)
import sys
sys.path.append("../src")

import pandas as pd
from data_quality import run_quality_checks
from kalman_filter import run_kalman_stage
from fuzzy_logic import run_fuzzy_stage
from backend_writer import make_records, send_to_backend
from load_data import load_raw

raw = load_raw()
result = run_fuzzy_stage(run_kalman_stage(run_quality_checks(raw)))

records = make_records(result)
print("Records:", len(records))
print()
print(result["health_status"].value_counts())
print()

# a graded reading
print(records[result.index.get_loc(result[result["health_status"] == "HEALTHY"].index[0])])
print()

# a reading in the dark: no health_score is sent (a missing value is left out, never 0)
print(records[result.index.get_loc(result[result["health_status"] == "NOT_GRADED"].index[0])])

Records: 196055

health_status
HEALTHY       172554
NOT_GRADED     15650
DEGRADED        6738
FAULTY          1113
Name: count, dtype: int64

{'timestamp_utc': '2026-09-08T00:50:20Z', 'record_id': 'M1-SITE-A-0000280', 'site_id': 'SITE-A', 'device_id': 'ESP32-A01', 'panel_id': 'PV-A01', 'quality_flag': 'OK', 'health_status': 'HEALTHY', 'voltage_v': 0.2247, 'current_ma': 1.8722, 'power_mw': 0.4206, 'panel_temp1_c': 24.2981, 'ambient_temp_c': 23.9996, 'irradiance_lux': 1017.709, 'dust_lpo_pct': 2.56, 'health_score': 84.7}

{'timestamp_utc': '2026-09-08T00:27:05Z', 'record_id': 'M1-SITE-A-0000001', 'site_id': 'SITE-A', 'device_id': 'ESP32-A01', 'panel_id': 'PV-A01', 'quality_flag': 'OK', 'health_status': 'NOT_GRADED', 'voltage_v': 0.012, 'current_ma': 0.1, 'power_mw': 0.0012, 'panel_temp1_c': 23.75, 'ambient_temp_c': 23.625, 'irradiance_lux': 18.08, 'dust_lpo_pct': 2.95}


In [2]:
# test: send only 5 records first
import backend_writer
print("Sending to:", backend_writer.BACKEND_URL)

send_to_backend(result.head(5))

Sending to: http://localhost:8000
Sent: 5 | accepted: 5 | rejected: 0


{'accepted': 5, 'rejected': 0}

In [ ]:
# send all records
send_to_backend(result)